# 42 · Padronização da escala das variáveis da tabela master

O k-means mede distância: uma variável em minutos (0 a 4.000) pesaria mil vezes mais que uma em
número de modalidades (0 a 5). A padronização põe todas na mesma escala.

- **Método**: z-score, `(valor − média) ÷ desvio-padrão`. Depois do tratamento de outliers (notebook 41)
  não sobram extremos que distorçam a média e o desvio.
- **População de referência**: média e desvio calculados nos assinantes ativos no corte (a mesma dos
  limites do notebook 41) e aplicados a todas as linhas. Entre os ativos, cada variável fica com média
  0 e desvio 1; nas outras contas, o valor diz a quantos desvios a conta está do assinante ativo típico.
- **Quais variáveis**: as numéricas candidatas ao k-means, na versão tratada quando ela existe
  (`_cap` ou `_log`). Ficam de fora as que repetem outra (`meses_de_casa`, `treinos_por_semana`,
  `ano_nascimento`), as substituídas por indicador e as de controle (`dias_historico_*`).
- **Indicadores de sim ou não** não são padronizados: já estão entre 0 e 1.
- **Vazios** continuam vazios. Colunas novas com sufixo `_z`; nada é alterado nas existentes.

Ordem de execução: 39 (monta) → 41 (trata outliers) → 42 (padroniza).

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
P = Path("../dados/processados"); S = Path("../saidas")

m = pd.read_parquet(P / "tabela_master.parquet")
print("tabela master lida:", m.shape)
VARS = ["idade_no_corte", "duracao_dias_log", "n_cobrancas_aprovadas_log", "receita_liquida",
        "treinos_7d", "treinos_30d", "treinos_90d_cap", "dias_ativos_90d", "minutos_90d_cap", "dias_sem_treinar_log",
        "n_tipos", "peso_tipo_principal", "treinos_corrida_log", "treinos_funcional_log", "treinos_caminhada_log", "treinos_forca_log", "treinos_yoga_log",
        "posts_90d", "n_seguidores", "n_seguindo", "n_mutuas",
        "dias_por_semana_declarado", "aderencia_meta_cap"]
falta = [c for c in VARS if c not in m.columns]
assert not falta, f"rode o notebook 41 antes: faltam {falta}"
ja = [c for c in m.columns if c.endswith("_z")]
m = m.drop(columns=ja)
print("colunas _z que já existiam e serão recriadas:", len(ja), "| colunas antes:", m.shape[1], "| variáveis a padronizar:", len(VARS))
orig = m.copy()
ref = m.ativa_no_corte
print("população de referência (assinantes ativos no corte):", int(ref.sum()))

tabela master lida: (63929, 95)
colunas _z que já existiam e serão recriadas: 0 | colunas antes: 95 | variáveis a padronizar: 23
população de referência (assinantes ativos no corte): 14753


## 1. Escala antes da padronização

In [2]:
antes = m.loc[ref, VARS].astype(float).describe().T[["count", "mean", "std", "min", "max"]]
antes["amplitude"] = antes["max"] - antes["min"]
print(antes.sort_values("amplitude", ascending=False))
print(); print("maior amplitude ÷ menor amplitude:", round(antes.amplitude.max() / antes.amplitude.min(), 1))

                               count      mean     std    min       max  amplitude
minutos_90d_cap           14,753.000 1,157.577 902.579  0.000 4,250.960  4,250.960
receita_liquida           14,753.000   362.578 215.880 33.915 1,007.130    973.215
treinos_90d_cap           14,753.000    25.090  18.560  0.000    84.000     84.000
peso_tipo_principal       13,999.000    62.377  19.672 20.000   100.000     80.000
dias_ativos_90d           14,753.000    20.201  13.558  0.000    73.000     73.000
treinos_30d               14,753.000     9.502   7.581  0.000    62.000     62.000
idade_no_corte            13,856.000    47.486  11.080 18.000    74.000     56.000
treinos_7d                14,753.000     2.230   2.236  0.000    17.000     17.000
n_seguidores              14,753.000     1.660   1.506  0.000    13.000     13.000
n_seguindo                14,753.000     1.671   1.495  0.000    11.000     11.000
n_mutuas                  14,753.000     1.241   1.262  0.000    10.000     10.000
post

## 2. Padronização (z-score com média e desvio dos assinantes ativos)

In [3]:
par = []
for c in VARS:
    s = m[c].astype(float)
    mu = s[ref].mean(); sd = s[ref].std(ddof=0)
    assert sd > 0, c
    m[c + "_z"] = ((s - mu) / sd).round(4)
    par.append({"variavel": c, "coluna_padronizada": c + "_z", "media_ativos": mu, "desvio_ativos": sd, "n_ativos_preenchidos": int(s[ref].notna().sum()), "n_preenchidos": int(s.notna().sum())})
par = pd.DataFrame(par).set_index("variavel")
print(par)
par.to_csv(S / "tabela_master_padronizacao.csv", encoding="utf-8-sig")

                                    coluna_padronizada  media_ativos  desvio_ativos  n_ativos_preenchidos  n_preenchidos
variavel                                                                                                                
idade_no_corte                        idade_no_corte_z        47.486         11.079                 13856          60095
duracao_dias_log                    duracao_dias_log_z         5.422          1.286                 14753          32583
n_cobrancas_aprovadas_log  n_cobrancas_aprovadas_log_z         1.386          0.815                 14753          32583
receita_liquida                      receita_liquida_z       362.578        215.873                 14753          32583
treinos_7d                                treinos_7d_z         2.230          2.236                 14753          54315
treinos_30d                              treinos_30d_z         9.502          7.581                 14753          53561
treinos_90d_cap                 

## 3. Conferências

In [4]:
Z = [c + "_z" for c in VARS]
print("linhas:", len(m), "(antes:", len(orig), ") | colunas:", orig.shape[1], "->", m.shape[1])
iguais = all(m[c].equals(orig[c]) for c in orig.columns); print("colunas existentes intactas:", iguais)
assert iguais and len(m) == len(orig) and m.user_id.is_unique
print("vazios preservados:", all(m[c + "_z"].isna().equals(m[c].isna()) for c in VARS))
d = m.loc[ref, Z].describe().T[["count", "mean", "std", "min", "max"]]
print(); print("assinantes ativos, depois da padronização (média 0 e desvio 1 esperados)"); print(d)
assert (d["mean"].abs() < 0.001).all() and ((d["std"] - 1).abs() < 0.001).all()
print(); print("amplitude depois: maior ÷ menor =", round((d["max"] - d["min"]).max() / (d["max"] - d["min"]).min(), 1))
print("valores acima de 4 desvios em módulo entre os ativos, por variável:")
print((m.loc[ref, Z].abs() > 4).sum()[lambda s: s > 0].to_frame("ativos"))
# a ordem das contas não muda: a padronização é uma mudança de escala
print("correlação de postos entre original e padronizada (deve ser 1):", round(min(m.loc[ref, c].astype(float).rank().corr(m.loc[ref, c + "_z"].astype(float).rank()) for c in VARS), 6))

linhas: 63929 (antes: 63929 ) | colunas: 95 -> 118
colunas existentes intactas: True
vazios preservados: True

assinantes ativos, depois da padronização (média 0 e desvio 1 esperados)
                                 count   mean   std    min   max
idade_no_corte_z            13,856.000 -0.000 1.000 -2.661 2.393
duracao_dias_log_z          14,753.000 -0.000 1.000 -4.215 1.773
n_cobrancas_aprovadas_log_z 14,753.000  0.000 1.000 -0.850 2.432
receita_liquida_z           14,753.000 -0.000 1.000 -1.522 2.986
treinos_7d_z                14,753.000  0.000 1.000 -0.997 6.604
treinos_30d_z               14,753.000 -0.000 1.000 -1.254 6.925
treinos_90d_cap_z           14,753.000 -0.000 1.000 -1.352 3.174
dias_ativos_90d_z           14,753.000 -0.000 1.000 -1.490 3.894
minutos_90d_cap_z           14,753.000 -0.000 1.000 -1.283 3.427
dias_sem_treinar_log_z      14,577.000  0.000 1.000 -1.080 3.119
n_tipos_z                   14,753.000 -0.000 1.000 -2.582 1.067
peso_tipo_principal_z       13,999.0

correlação de postos entre original e padronizada (deve ser 1): 1.0


In [5]:
print("fora da população de referência: média das variáveis padronizadas por situação (em desvios do assinante ativo típico)")
print(m.groupby("situacao")[Z].mean().T.round(2))
print()
print("linhas com todas as", len(VARS), "variáveis preenchidas, por situação:")
comp = m[Z].notna().all(axis=1)
t = pd.crosstab(m.situacao, comp.map({True: "completa", False: "com vazio"}), margins=True); t["% completa"] = t["completa"] / t["All"] * 100; print(t)
print(); print("variáveis que mais deixam vazio entre os ativos:"); print(m.loc[ref, Z].isna().sum()[lambda s: s > 0].sort_values(ascending=False).to_frame("ativos sem valor"))

fora da população de referência: média das variáveis padronizadas por situação (em desvios do assinante ativo típico)
situacao                     assinante ativo  em teste grátis  ex-assinante  nunca iniciou teste  teste expirou, não assinou
idade_no_corte_z                      -0.000           -0.020         0.040               -0.020                       0.000
duracao_dias_log_z                    -0.000              NaN        -0.000                  NaN                         NaN
n_cobrancas_aprovadas_log_z            0.000              NaN         0.080                  NaN                         NaN
receita_liquida_z                     -0.000              NaN        -0.740                  NaN                         NaN
treinos_7d_z                           0.000           -0.200        -0.750               -0.970                      -0.970
treinos_30d_z                         -0.000           -1.020        -0.720               -1.220                      -1.220
treinos

## 4. Dicionário e gravação

In [6]:
dic = pd.read_csv(S / "tabela_master_dicionario.csv")
dic = dic[~dic.coluna.str.endswith("_z")][["coluna", "grupo", "descricao", "fonte"]]
def br(v): return f"{v:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
extra = [(c + "_z", "padronizada para k-means", f"{c} em z-score: (valor − {br(par.media_ativos[c])}) ÷ {br(par.desvio_ativos[c])}, média e desvio dos assinantes ativos no corte", c) for c in VARS]
dic = pd.concat([dic, pd.DataFrame(extra, columns=["coluna", "grupo", "descricao", "fonte"])], ignore_index=True)
assert list(dic.coluna) == list(m.columns), "dicionário e tabela fora de ordem"
dic["tipo"] = [str(m[c].dtype) for c in dic.coluna]
dic["preenchidas"] = [int(m[c].notna().sum()) for c in dic.coluna]
dic["% preenchido"] = (dic.preenchidas / len(m) * 100).round(1)
dic["% preenchido (ativos)"] = [round(m.loc[m.ativa_no_corte, c].notna().mean() * 100, 1) for c in dic.coluna]
m.to_parquet(P / "tabela_master.parquet", index=False)
m.to_csv(S / "tabela_master.csv", index=False, encoding="utf-8-sig")
dic.to_csv(S / "tabela_master_dicionario.csv", index=False, encoding="utf-8-sig")
print("gravado:", m.shape, "| colunas por grupo:", dic.grupo.value_counts().to_dict())

gravado: (63929, 118) | colunas por grupo: {'padronizada para k-means': 23, 'uso': 22, 'comportamento': 18, 'tratada para k-means': 15, 'entrada': 13, 'saída': 9, 'perfil': 7, 'pagamento': 4, 'identificação': 3, 'suporte': 3, 'campanha': 1}
